# STAT 626 — Three-slice MRI segmentation with availability-masked attention

Start with **QUICK_CHECK** and run sections 1–5 to view the EDA figures. Continue through section 9 for a six-patient pipeline check. Switch to **STUDY_100_AMCA** and rerun sections 2–5 for the full 100-patient audit. Continue through sections 6–8 to train M0, M1 and M2. Section 9 evaluates the held-out groups after model selection.

Each model receives three neighboring axial slices per MRI sequence plus four availability maps. The shared 2.5D U-Net predicts the center slice with 2D convolutions. Cross-attention uses the available-sequence mean as the query at each pixel. M0 uses complete inputs during training, M1 adds supervised sequence dropout, and M2 adds consistency.

Figures and checkpoints go to your Drive. The full study uses a 70/15/15 split by patient group and up to ten epochs at width 16. Use a GPU runtime for full training. Displaying saved PNGs or performing EDA can use CPU.

## 1. Install dependencies and check the runtime

Use the existing PyTorch installation. The notebook detects the available device.

In [ ]:
import os, sys, subprocess, importlib.metadata
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["MPLBACKEND"] = "Agg"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "nibabel>=5.2,<6", "numpy>=1.26,<3", "scipy>=1.11,<2",
                "matplotlib>=3.8,<4"], check=True)
import torch, numpy as np
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.3"):
    raise RuntimeError("Use a current Colab GPU runtime with PyTorch >= 2.3.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
ENVIRONMENT = {"python": sys.version.split()[0], "torch": str(torch.__version__),
               "cuda": torch.version.cuda, "device": DEVICE,
               **{name: importlib.metadata.version(name) for name in ("numpy", "nibabel", "scipy", "matplotlib")}}

## 2. Mount Drive and choose the run

Place the existing **BraTS100_2D.zip** in **My Drive/STAT626**, or point DATA_SOURCE to its full Drive path. The earlier 200-patient archive, the original shard manifest directory, and an extracted BraTS folder are also supported.

| Profile | Image QC | Models | Width | Epochs |
|---|---:|---|---:|---:|
| QUICK_CHECK | 6 selected exams | M0, M1, M2 | 4 | 1 |
| STUDY_100_AMCA | 100 selected exams | M0, M1, M2 | 16 | 10 |

Both profiles use seed 11 for training, seed 626 for patient grouping and splits, and a fixed consistency weight of 0.1 for M2. Results go to separate profile folders in the new STUDY_BASE directory.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, copy

PROFILE = "QUICK_CHECK" # @param ["QUICK_CHECK", "STUDY_100_AMCA"]
DATA_SOURCE = "/content/drive/MyDrive/STAT626" # @param {type:"string"}
STUDY_BASE = "/content/drive/MyDrive/STAT626_100patients_AMCA_v1" # @param {type:"string"}
PATIENT_MAP = "" # @param {type:"string"}
LOCAL_DATA_ROOT = "/content/stat626_100_2d_data" # @param {type:"string"}
STUDY_DIR = Path(STUDY_BASE) / PROFILE.lower()
CODE_DIR = Path("/content/stat626_100_amca_code")
if not Path(DATA_SOURCE).exists():
    raise FileNotFoundError(f"Correct DATA_SOURCE or upload the subset ZIP first: {DATA_SOURCE}")
print("Profile:", PROFILE, "Persistent outputs:", STUDY_DIR)

## 3. Load the embedded project code

This extracts the Python scripts and the frozen 100-patient selection bundled with this notebook.

In [ ]:
# @title Restore three-slice AMCA source and the frozen 100-group selection
import base64, hashlib, io, zipfile, importlib
CODE_B64 = "..."
payload = base64.b64decode(CODE_B64)
assert hashlib.sha256(payload).hexdigest() == "0e608fe1c6fcf5e6451ba649fae74ecf90c3c4780743cb2390a3efaf746c4d19"
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == {'design_network.py', 'subset_selection.json', 'select_subset.py', 'train_model.py', 'eda.py', 'colab_workflow.py', 'verify_amca.py', 'requirements_stat626.txt'}
    archive.extractall(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for module_name in ("eda", "design_network", "train_model", "colab_workflow", "select_subset"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
import colab_workflow as workflow
from select_subset import stage_subset
SETTINGS = copy.deepcopy(workflow.PROFILES[PROFILE])
SETTINGS.update(data_source=DATA_SOURCE, local_data_root=LOCAL_DATA_ROOT,
                expected_groups=100, patient_map=PATIENT_MAP,
                patient_map_sha256=workflow.digest(PATIENT_MAP) if PATIENT_MAP else None)
workflow.lock_study(STUDY_DIR, CODE_DIR, PROFILE, SETTINGS, ENVIRONMENT)
print(json.dumps(SETTINGS, indent=2))
selection = json.loads((CODE_DIR / "subset_selection.json").read_text())
print("Frozen selected groups:", selection["selected_patient_groups"])

## 4. Stage only the selected examinations
This keeps 100 examinations on local runtime storage. The original dataset remains unchanged.
For the original shard format, SHA-256 hashes are verified before extracting selected files.
For a single ZIP, ZIP CRC is checked during extraction. The selection JSON fixes case IDs and sizes.
After a reset, rerun staging and training cells; your trained epoch checkpoints remain on Drive.


In [ ]:
DATA_ROOT = stage_subset(DATA_SOURCE, LOCAL_DATA_ROOT, CODE_DIR / "subset_selection.json")
print("Selected data staged:", DATA_ROOT)


## 5. Run EDA, quality control, and freeze patient splits
Saved audits are reused with integrity checks. A partial QUICK_CHECK audit is never
used as the full study audit. Inspect data counts, naturally absent files, dimensions,
and the distinction between acquired scans and simulated missing-input views.


In [ ]:
EDA_DIR, EDA_SUMMARY = workflow.audit(CODE_DIR, DATA_ROOT, STUDY_DIR, SETTINGS, PATIENT_MAP)
for key in ("audit_scope", "data_volume", "retained_scan_folders", "retained_patient_groups",
            "retained_slice_dimensions", "split_scan_counts", "split_patient_counts"):
    print(key, ":", json.dumps(EDA_SUMMARY.get(key), indent=2))

from IPython.display import display, Image
pictures = sorted(EDA_DIR.glob("*.png"))
print(f"Saved {len(pictures)} figures to {EDA_DIR}")
for picture in pictures:
    print(picture.name)
    display(Image(filename=str(picture), width=900))

## Availability-Masked Cross-Attention Fusion

For target slice $z$, each modality supplies $x_m=(z-1,z,z+1)$ on a $240\times240$ grid. At the ends of the volume, repeat the closest valid slice. Each modality stem produces $F_m=\phi_m(x_m)$. An availability flag $a_m$ gates that modality.

$$
\bar F_u=\frac{\sum_m a_m F_m(u)}{\sum_m a_m},\qquad
q_u=W_q\bar F_u,\quad k_{m,u}=W_k F_m(u)+e_m,\quad v_{m,u}=W_v F_m(u).
$$

For each of four heads, $s_{m,u}=q_u^\top k_{m,u}/\sqrt{d}$. Set unavailable scores to $-\infty$ and take softmax across modalities. The weighted sum of values in each head is concatenated into $z_u$. The residual $F_u^\star=\bar F_u+W_o z_u$ is joined with the four availability maps before the U-Net. At base width 16, the stem has eight features and each head has dimension two.

All three regimes share exactly this architecture. The segmentation loss is cross-entropy plus foreground soft Dice. M0 optimizes the complete-view loss, M1 averages the complete and single-missing losses, and M2 adds $0.1\,D_{\mathrm{KL}}(p_{\mathrm{full}}\Vert p_{\mathrm{missing}})$, averaged over the fixed brain mask. The complete-view distribution is detached for the KL term. When a sequence is withheld, its three images and availability flag are zeroed.

Section 10 provides a separate M1 ablation that replaces the attention residual with concatenation of the gated stem features. It uses the same patient split and supervision.

## 6. Check the shared network on a real batch

The checks compare the code against the attention equations, verify missing-sequence masking and test the three-slice ordering. The preflight runs a forward and backward pass on a batch of archived images.

In [ ]:
workflow.run([sys.executable, "-u", CODE_DIR / "design_network.py", "--self-test"])
PREFLIGHT = workflow.preflight(EDA_DIR, SETTINGS, device=DEVICE)
workflow.save_json(STUDY_DIR / "preflight.json", {"environment": ENVIRONMENT, "checks": PREFLIGHT})
print("Preflight passed.")

## 7. Train or resume M0, M1, and M2

The full run samples 16 target slices per training patient per epoch, with up to 10 epochs per model. Validation stacks predictions for all axial slices under complete input and the four single-missing conditions. A restart can resume from the last saved epoch. Rerun sections 1–7 with the same profile and settings after a reset.

In [ ]:
TRAIN_COMMAND = workflow.train_command(CODE_DIR, EDA_DIR, STUDY_DIR, SETTINGS, device=DEVICE)
workflow.run(TRAIN_COMMAND)
EXPERIMENT = json.loads((STUDY_DIR / "models/experiment.json").read_text())
print("Training status:", EXPERIMENT["status"])
print("Selected consistency weights:", EXPERIMENT.get("selected_consistency_weights"))

## 8. Display saved training and validation curves

The training objectives differ across models. Use validation Dice to compare their segmentation performance.

In [ ]:
import matplotlib.pyplot as plt
histories = sorted((STUDY_DIR / "models/final_runs").glob("*/history.json"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for path in histories:
    history = json.loads(path.read_text())
    axes[0].plot([r["epoch"] for r in history], [r["loss"] for r in history], marker=".", label=path.parent.name)
    valid = [r for r in history if "validation_mean_dice" in r]
    axes[1].plot([r["epoch"] for r in valid], [r["validation_mean_dice"] for r in valid], marker=".", label=path.parent.name)
for ax, title in zip(axes, ("Training loss", "Validation mean Dice")):
    ax.set(xlabel="Epoch", title=title)
    ax.grid(alpha=.2)
    if histories:
        ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(STUDY_DIR / "training_curves.png", dpi=160)
plt.close(fig)
display(Image(filename=str(STUDY_DIR / "training_curves.png"), width=900))

## 9. Evaluate the held-out patients

QUICK_CHECK scores one held-out exam to verify execution. For the 100-patient protocol, turn on RUN_HELD_OUT_TEST once training and model choices are frozen. The primary contrast is M2 minus M1 averaged over WT, TC and ET and all four single-missing conditions. M1 minus M0 describes the effect of training with sequence dropout. Both use paired patient scores.

Turn on SAVE_ALL_PREDICTIONS to save NIfTI masks and visual comparisons. With this switch enabled, the cell also displays example PNGs. A successfully completed evaluation is reused on later runs.

In [ ]:
RUN_HELD_OUT_TEST = False # @param {type:"boolean"}
SAVE_ALL_PREDICTIONS = False # @param {type:"boolean"}
EVALUATION_DIR = None
if PROFILE == "QUICK_CHECK" or RUN_HELD_OUT_TEST:
    EVALUATION_DIR = workflow.evaluate(CODE_DIR, EDA_DIR, STUDY_DIR, SETTINGS,
                                      save_predictions=(PROFILE == "QUICK_CHECK" or SAVE_ALL_PREDICTIONS), device=DEVICE)
    print((EVALUATION_DIR / "evaluation_summary.json").read_text())
    import csv
    with (EVALUATION_DIR / "metric_summary.csv").open() as handle:
        for row in csv.DictReader(handle):
            print(row)
    examples = sorted(EVALUATION_DIR.glob("predictions/**/qualitative_comparison.png"))
    for path in examples[:3]:
        display(Image(filename=str(path), width=1000))
else:
    print("Training complete. Held-out test remains unused. Review validation and freeze choices first.")

The results folder printed below contains the audit, training histories and the evaluation location. The QUICK_CHECK split contains one held-out patient. Population comparisons require the full study.

In [ ]:
print("Persistent results:", STUDY_DIR)
print("Profile:", PROFILE)
print("QUICK_CHECK results are pipeline checks only." if PROFILE == "QUICK_CHECK" else "Report full study metrics only after locked evaluation.")


## 10. Optional M1 channel-concatenation ablation

This comparison trains M1 with gated concatenated features and pairs its held-out scores with the main M1 attention scores. Choose the ablation before viewing held-out results if you plan to interpret it as a prespecified comparison. Enable evaluation after both M1 configurations are trained. All outputs go to the profile's independent concatenation folder.

In [ ]:
RUN_CONCATENATION = False # @param {type:"boolean"}
EVALUATE_CONCATENATION = False # @param {type:"boolean"}
concat_settings = dict(SETTINGS, models=["M1"], fusion_mode="concatenation")
concat_dir = STUDY_DIR / "concatenation_ablation"
if RUN_CONCATENATION or EVALUATE_CONCATENATION:
    workflow.lock_study(concat_dir, CODE_DIR, PROFILE, concat_settings, ENVIRONMENT)
if RUN_CONCATENATION:
    workflow.run(workflow.train_command(CODE_DIR, EDA_DIR, concat_dir, concat_settings, device=DEVICE))
if EVALUATE_CONCATENATION:
    from train_model import read_csv, paired_contrast
    main_location = STUDY_DIR / "evaluation_location.json"
    if not main_location.is_file():
        raise RuntimeError("Complete section 9 before the paired concatenation comparison.")
    attention = Path(json.loads(main_location.read_text())["path"])
    concat = workflow.evaluate(CODE_DIR, EDA_DIR, concat_dir, concat_settings, device=DEVICE)
    rows = [
        dict(row, model=label)
        for directory, label in ((attention, "Attention M1"), (concat, "Concatenation M1"))
        for row in read_csv(directory / "patient_scores.csv")
        if row["model"] == "M1"
    ]
    conditions = {"complete": ("complete",), "single_missing": ("t1n", "t1c", "t2w", "t2f")}
    comparison = {}
    for key, values in conditions.items():
        comparison[key], _ = paired_contrast(rows, "Attention M1", "Concatenation M1",
                                              values, SETTINGS["seeds"], 2000, 626)
    workflow.save_json(concat_dir / "attention_minus_concatenation.json", comparison)
    print(json.dumps(comparison, indent=2))
if not RUN_CONCATENATION and not EVALUATE_CONCATENATION:
    print("Optional concatenation ablation is disabled.")